# 14.7 Trees and Binary Search Trees

**Prerequisites:** 14.4 Linked Lists, 14.5 Stacks and Queues, 14.1 Complexity Analysis  
**Target:** Python 3.12+ (notes flag 3.13/3.14 differences)

### What you'll learn
- Tree vocabulary - root, leaf, depth, height, and why they get confused
- The four traversals, recursively **and** iteratively
- 🔴 Why **in-order** on a BST comes out sorted - and what that buys you
- BST search, insert and delete, including the three-case delete
- 🔴 **Degeneration**: how a BST becomes a linked list, measured
- Balanced trees - AVL and red-black, and what self-balancing costs
- 🔴 **Validating a BST** - the trap almost everyone falls into
- Height, diameter, lowest common ancestor
- Interview questions, worked

---

## From lists to trees

A linked list (**14.4**) is a chain: each node points to **one** successor. A tree is the same idea with **several** successors — and that single change buys you O(log n) instead of O(n), provided the tree stays balanced.

```
                  ┌────┐
                  │ 50 │  <- root (depth 0)
                  └────┘
                 /      \
            ┌────┐      ┌────┐
            │ 30 │      │ 70 │      <- depth 1
            └────┘      └────┘
           /     \           \
      ┌────┐   ┌────┐      ┌────┐
      │ 20 │   │ 40 │      │ 80 │   <- depth 2, all LEAVES
      └────┘   └────┘      └────┘
```

| Term | Means |
|---|---|
| **root** | the single node with no parent |
| **leaf** | a node with no children |
| **depth** of a node | edges from the **root down** to it |
| **height** of a node | edges from it **down to its deepest leaf** |
| **height of the tree** | the height of the root |
| **subtree** | any node, taken with everything below it |

🔴 **Depth and height are measured in opposite directions** and are constantly confused. A leaf has height 0; the root has depth 0. In the tree above the root has height 2 and node 20 has depth 2.

> **Why trees matter:** a balanced binary tree of height h holds up to 2^(h+1)−1 nodes. Turn that round: a million nodes fit in a tree of height **20**. Every search, insert and delete walks one root-to-leaf path — so all three are O(log n).

In [ ]:
class TreeNode:
    """A binary tree node: a value and up to two children."""

    __slots__ = ("value", "left", "right")

    def __init__(self, value, left=None, right=None):
        self.value = value
        self.left = left
        self.right = right

    def __repr__(self):
        return f"TreeNode({self.value!r})"


def build_sample():
    """The tree drawn above."""
    return TreeNode(50,
                    TreeNode(30, TreeNode(20), TreeNode(40)),
                    TreeNode(70, None, TreeNode(80)))


def height(node):
    """Edges to the deepest leaf. An empty tree is -1 so a leaf is 0."""
    if node is None:
        return -1
    return 1 + max(height(node.left), height(node.right))


def count_nodes(node):
    return 0 if node is None else 1 + count_nodes(node.left) + count_nodes(node.right)


def count_leaves(node):
    if node is None:
        return 0
    if node.left is None and node.right is None:
        return 1
    return count_leaves(node.left) + count_leaves(node.right)


root = build_sample()
print("root        :", root)
print("nodes       :", count_nodes(root))
print("leaves      :", count_leaves(root))
print("height      :", height(root), " (edges from root to deepest leaf)")
print("height of 20:", height(root.left.left), " (a leaf)")
print("\nNotice how naturally recursion fits: a tree IS a value plus two")
print("smaller trees, so almost every tree function is 'handle None, then")
print("combine the results from left and right'.")

## The four traversals

There are only two questions: **do you go depth-first or breadth-first**, and if depth-first, **when do you visit the node relative to its children**.

```
        50
       /  \
     30    70
    /  \     \
  20    40    80
```

| Traversal | Order | Result | Use for |
|---|---|---|---|
| **Pre-order** | node, left, right | 50 30 20 40 70 80 | copying a tree, serialising |
| **In-order** | left, node, right | 20 30 40 50 70 80 | 🔴 **sorted output from a BST** |
| **Post-order** | left, right, node | 20 40 30 80 70 50 | deleting, evaluating expressions |
| **Level-order** | breadth-first | 50 30 70 20 40 80 | shortest path, printing by level |

The first three differ by **one line's position**. That is genuinely all.

> **How to remember:** the prefix says when the **node** is visited. *Pre*-order visits it before its children, *in*-order between them, *post*-order after.

In [ ]:
from collections import deque


def preorder(node, out=None):
    out = [] if out is None else out
    if node is not None:
        out.append(node.value)        # <- node FIRST
        preorder(node.left, out)
        preorder(node.right, out)
    return out


def inorder(node, out=None):
    out = [] if out is None else out
    if node is not None:
        inorder(node.left, out)
        out.append(node.value)        # <- node BETWEEN
        inorder(node.right, out)
    return out


def postorder(node, out=None):
    out = [] if out is None else out
    if node is not None:
        postorder(node.left, out)
        postorder(node.right, out)
        out.append(node.value)        # <- node LAST
    return out


def level_order(root):
    """Breadth-first, using a QUEUE - the deque from 14.5."""
    if root is None:
        return []
    out, queue = [], deque([root])
    while queue:
        node = queue.popleft()        # O(1) - never list.pop(0)
        out.append(node.value)
        if node.left:
            queue.append(node.left)
        if node.right:
            queue.append(node.right)
    return out


root = build_sample()
print("pre-order  :", preorder(root))
print("in-order   :", inorder(root), " <- SORTED, because this is a BST")
print("post-order :", postorder(root))
print("level-order:", level_order(root))
print("\n🔴 Depth-first uses a STACK (the call stack, or an explicit one).")
print("   Breadth-first uses a QUEUE. That one difference is what makes")
print("   them explore in completely different orders (14.5).")

### Level-order, level by level

The variant that actually gets asked: not one flat list, but a list per level. The trick is to **record the queue's length before draining it** — that count is exactly the number of nodes on the current level.

```
    while queue:
        width = len(queue)          <- everything currently queued is one level
        for _ in range(width):
            ...process, enqueue children...
```

This is the same BFS that finds shortest paths in graphs (**14.9**) — a tree is just a graph with no cycles.

In [ ]:
def level_order_grouped(root):
    """One list per level. O(n) time, O(width) space."""
    if root is None:
        return []
    levels, queue = [], deque([root])
    while queue:
        width = len(queue)                    # 🔴 snapshot BEFORE draining
        level = []
        for _ in range(width):
            node = queue.popleft()
            level.append(node.value)
            if node.left:
                queue.append(node.left)
            if node.right:
                queue.append(node.right)
        levels.append(level)
    return levels


root = build_sample()
for depth, level in enumerate(level_order_grouped(root)):
    print(f"  depth {depth}: {level}")

print("\nfrom which several classic questions fall out immediately:")
levels = level_order_grouped(root)
print("  max depth        :", len(levels) - 1)
print("  right-side view  :", [level[-1] for level in levels])
print("  widest level     :", max(len(level) for level in levels))
print("  zigzag order     :", [level if i % 2 == 0 else level[::-1]
                              for i, level in enumerate(levels)])

### Iterative traversal, and why you would bother

Recursion is clearer, and it costs O(h) stack space. A **degenerate** tree of 10,000 nodes has height 10,000 — and Python's recursion limit is about 1,000 (**14.1**).

The iterative versions move that stack onto the heap, exactly as **14.5** did for the call stack.

🔴 **Pre-order iteratively has one counter-intuitive detail:** push the **right** child first, so the left is popped first. A stack reverses whatever you put in.

In-order iteratively is the one worth memorising — walk left as far as possible pushing as you go, then pop, visit, and turn right.

In [ ]:
def preorder_iterative(root):
    if root is None:
        return []
    out, stack = [], [root]
    while stack:
        node = stack.pop()
        out.append(node.value)
        if node.right:
            stack.append(node.right)   # 🔴 RIGHT first...
        if node.left:
            stack.append(node.left)    # ...so LEFT is popped first
    return out


def inorder_iterative(root):
    """Walk left pushing, then pop-visit-turn-right. Worth memorising."""
    out, stack, node = [], [], root
    while stack or node is not None:
        while node is not None:        # go as far left as possible
            stack.append(node)
            node = node.left
        node = stack.pop()             # nothing left of here
        out.append(node.value)
        node = node.right              # now handle the right subtree
    return out


root = build_sample()
print("pre-order  recursive:", preorder(root))
print("pre-order  iterative:", preorder_iterative(root))
print("in-order   recursive:", inorder(root))
print("in-order   iterative:", inorder_iterative(root))

# A degenerate tree: every node has only a right child - a linked list.
deep = TreeNode(0)
node = deep
for i in range(1, 3_000):
    node.right = TreeNode(i)
    node = node.right

print(f"\na degenerate tree of 3,000 nodes has height {3_000 - 1}:")
try:
    inorder(deep)
    print("  recursive: fine")
except RecursionError:
    print("  recursive: RecursionError - height exceeded the stack limit")
print("  iterative:", len(inorder_iterative(deep)), "nodes visited, no problem")

---

# Binary Search Trees

A binary tree with one extra rule, applied at **every** node:

> **Everything in the left subtree is smaller. Everything in the right subtree is larger.**

```
        50            search for 40:
       /  \             40 < 50  -> go left
     30    70           40 > 30  -> go right
    /  \     \          found
  20    40    80
```

Each comparison discards **half** the remaining tree — the same halving as binary search (**14.11**), which is where O(log n) comes from.

| Operation | Balanced | 🔴 Degenerate |
|---|---|---|
| Search | O(log n) | O(n) |
| Insert | O(log n) | O(n) |
| Delete | O(log n) | O(n) |
| In-order traversal | O(n) | O(n) |

**The word "every" in the rule matters.** It is not enough that a node is bigger than its left child; it must be bigger than *everything* in its left subtree. That is the validation trap below.

In [ ]:
def bst_insert(root, value):
    """Insert, returning the (possibly new) root. Duplicates ignored."""
    if root is None:
        return TreeNode(value)
    if value < root.value:
        root.left = bst_insert(root.left, value)
    elif value > root.value:
        root.right = bst_insert(root.right, value)
    return root                        # equal: ignore


def bst_search(root, value):
    """Return (found, comparisons) so we can see the halving."""
    comparisons = 0
    node = root
    while node is not None:
        comparisons += 1
        if value == node.value:
            return True, comparisons
        node = node.left if value < node.value else node.right
    return False, comparisons


def bst_min(node):
    while node.left is not None:       # leftmost node
        node = node.left
    return node


root = None
for value in (50, 30, 70, 20, 40, 80, 60):
    root = bst_insert(root, value)

print("in-order  :", inorder(root), " <- sorted, for free")
print("minimum   :", bst_min(root).value)
print("height    :", height(root))
print()
for target in (40, 60, 99):
    found, comparisons = bst_search(root, target)
    print(f"  search {target:>3}: {'found' if found else 'absent':<7} "
          f"in {comparisons} comparisons")

print("\n7 nodes, at most 3 comparisons. A balanced tree of 1,000,000")
print("nodes needs at most 20 - that is the whole appeal.")

### Deleting from a BST - three cases

The only fiddly BST operation, and a standard interview question.

| The node has | Do |
|---|---|
| **no children** | remove it |
| **one child** | replace it with that child |
| **two children** | 🔴 replace its value with its **in-order successor**, then delete that successor |

**Why the in-order successor** (the smallest value in the right subtree)? Because it is the *next* value in sorted order — larger than everything left, smaller than everything else right. Putting it in the gap keeps the BST rule intact everywhere.

The in-order **predecessor** (largest in the left subtree) works just as well; pick one and be consistent.

> The successor is guaranteed to have **at most one child** — it is the leftmost node of the right subtree, so it cannot have a left child. That is why the recursion terminates.

In [ ]:
def bst_delete(root, value):
    """Delete a value, returning the new root."""
    if root is None:
        return None
    if value < root.value:
        root.left = bst_delete(root.left, value)
    elif value > root.value:
        root.right = bst_delete(root.right, value)
    else:
        # case 1 and 2: no children, or exactly one
        if root.left is None:
            return root.right          # covers 'no children' too (None)
        if root.right is None:
            return root.left
        # case 3: two children -> take the in-order successor
        successor = bst_min(root.right)
        root.value = successor.value
        root.right = bst_delete(root.right, successor.value)
    return root


def fresh_tree():
    tree = None
    for value in (50, 30, 70, 20, 40, 60, 80):
        tree = bst_insert(tree, value)
    return tree


for target, label in ((20, "leaf, no children"),
                     (70, "two children"),
                     (50, "the root, two children")):
    tree = fresh_tree()
    before = inorder(tree)
    tree = bst_delete(tree, target)
    after = inorder(tree)
    print(f"  delete {target:>3} ({label})")
    print(f"    {before} -> {after}")
    print(f"    still sorted: {after == sorted(after)}")

tree = fresh_tree()
tree = bst_delete(tree, 999)
print(f"\n  deleting an absent value is harmless: {inorder(tree)}")

## 🔴 Degeneration - the BST's fatal flaw

Insert **already-sorted** data into a plain BST and every value goes to the right of the last. The tree becomes a linked list.

```
   insert 10, 20, 30, 40, 50 in order:

        10
          \
           20                height = n-1, not log n
             \
              30             every operation is now O(n)
                \
                 40
                   \
                    50
```

This is not a rare edge case — **sorted input is extremely common**. Ids, timestamps, anything coming out of a database with an `ORDER BY`. A BST fed sorted data is the textbook example of an algorithm whose average case and real-world case are nothing alike.

The next cell measures it.

In [ ]:
import math
import random

N = 2_000


def bst_insert_iterative(root, value):
    """Insert without recursion.

    🔴 Why this exists: `bst_insert` recurses once per level, so on a
    DEGENERATE tree it recurses once per node. Building the 2,000-node
    degenerate tree below with the recursive version raises
    RecursionError before the measurement can even run - the recursive
    insert cannot construct the very case it is meant to illustrate.
    """
    if root is None:
        return TreeNode(value)
    node = root
    while True:
        if value < node.value:
            if node.left is None:
                node.left = TreeNode(value)
                return root
            node = node.left
        elif value > node.value:
            if node.right is None:
                node.right = TreeNode(value)
                return root
            node = node.right
        else:
            return root                      # duplicate: ignore


def height_iterative(root):
    """Recursion would blow the stack on the degenerate tree too."""
    if root is None:
        return -1
    best = -1
    stack = [(root, 0)]
    while stack:
        node, depth = stack.pop()
        best = max(best, depth)
        if node.left:
            stack.append((node.left, depth + 1))
        if node.right:
            stack.append((node.right, depth + 1))
    return best


# ---- sorted input: the worst case ----
degenerate = None
for value in range(N):
    degenerate = bst_insert_iterative(degenerate, value)

# ---- shuffled input: near-best case ----
rng = random.Random(15)
shuffled_values = list(range(N))
rng.shuffle(shuffled_values)
balanced = None
for value in shuffled_values:
    balanced = bst_insert_iterative(balanced, value)

print(f"{N:,} nodes inserted two ways:\n")
print(f"  sorted input   -> height {height_iterative(degenerate):>6,}")
print(f"  shuffled input -> height {height_iterative(balanced):>6,}")
print(f"  perfectly balanced would be {int(math.log2(N)):>2}")

print("\nsearching for the last value:")
_, degenerate_steps = bst_search(degenerate, N - 1)
_, balanced_steps = bst_search(balanced, N - 1)
print(f"  degenerate : {degenerate_steps:>6,} comparisons   O(n)")
print(f"  shuffled   : {balanced_steps:>6,} comparisons   O(log n)")
print(f"  ratio      : {degenerate_steps / balanced_steps:>6.0f}x")
print("\n🔴 Same data, same code. Only the INSERTION ORDER differed, and the")
print("   structure collapsed from a tree into a linked list (14.4).")
print("\n🔴 Note the insert above had to be ITERATIVE. The recursive one")
print("   raises RecursionError building this tree - O(height) stack, and")
print("   here height IS n. Degeneration breaks more than lookup speed.")

## Self-balancing trees

The fix is a tree that **restructures itself** on insert and delete to keep its height O(log n).

| Tree | Balance rule | Character |
|---|---|---|
| **AVL** | heights of any node's subtrees differ by ≤ 1 | stricter; faster lookups, more rotations |
| **Red-black** | colouring rules bounding the longest path to 2× the shortest | looser; fewer rotations, faster writes |
| **B-tree** | many children per node, all leaves at the same depth | 🔴 what databases and filesystems use (**10.1**) |

All of them work by **rotation** — a local rearrangement that changes the height without breaking the ordering:

```
        30                 20
       /          ->      /  \
     20                 10    30
    /
  10        right rotation: in-order is 10 20 30 both before and after
```

> **Do you need to implement one?** Almost never — and interviewers rarely ask, because a correct red-black insert is 100+ lines. **Know what problem they solve, name one, and explain a rotation.** That is the expected depth.

🔴 **Python has no built-in balanced tree.** Use a `dict` when you need O(1) lookup, or `sortedcontainers` (third-party) when you need sorted order with fast insertion. A sorted `list` plus `bisect` gives O(log n) search but O(n) insert (**14.2**).

In [ ]:
def rotate_right(root):
    """The single rotation every balanced tree is built from."""
    pivot = root.left
    root.left = pivot.right            # pivot's right subtree moves across
    pivot.right = root
    return pivot                       # pivot is the new root


# 10 -> 20 -> 30 leaning left
leaning = TreeNode(30, TreeNode(20, TreeNode(10)))
print("before rotation:")
print("  in-order:", inorder(leaning), " height:", height(leaning))

rotated = rotate_right(leaning)
print("after rotation :")
print("  in-order:", inorder(rotated), " height:", height(rotated))
print("\n🔴 The in-order traversal is IDENTICAL - the ordering is preserved.")
print("   Only the height changed, from 2 to 1. That is the entire trick,")
print("   applied automatically after every insert and delete.")

## 🔴 Validating a BST - the classic trap

*"Is this a valid binary search tree?"* The obvious answer is wrong, and it is wrong in a way that passes small tests.

**The wrong version:** check each node against its immediate children.

```
        10
       /  \
      5    15
          /  \
         6    20        <- 6 < 10, so it does NOT belong in the right subtree
```

Every parent/child pair here is fine: 6 < 15 ✓, and 15 > 10 ✓. But 6 is in the right subtree of 10, and 6 < 10 — so this is **not** a BST.

**The fix:** carry a valid **range** down the tree. Going left tightens the upper bound; going right tightens the lower bound.

```
    validate(node, low, high):
        low < node.value < high
        validate(node.left,  low,        node.value)
        validate(node.right, node.value, high)
```

**Alternative:** an in-order traversal must be strictly increasing. Equally valid, arguably clearer, and it needs O(n) space unless you track only the previous value.

In [ ]:
def is_bst_wrong(node):
    """The seductive, incorrect version: checks only immediate children."""
    if node is None:
        return True
    if node.left and node.left.value >= node.value:
        return False
    if node.right and node.right.value <= node.value:
        return False
    return is_bst_wrong(node.left) and is_bst_wrong(node.right)


def is_bst(node, low=float("-inf"), high=float("inf")):
    """Correct: every node must fall inside a range narrowed by its ancestors."""
    if node is None:
        return True
    if not low < node.value < high:
        return False
    return (is_bst(node.left, low, node.value)
            and is_bst(node.right, node.value, high))


def is_bst_inorder(root):
    """Equally correct: in-order must be strictly increasing."""
    values = inorder(root)
    return all(a < b for a, b in zip(values, values[1:]))


# the counter-example from the markdown above
sneaky = TreeNode(10, TreeNode(5), TreeNode(15, TreeNode(6), TreeNode(20)))
valid = fresh_tree()

print(f"{'tree':<26}{'wrong':>8}{'range':>8}{'in-order':>10}")
print("-" * 52)
for label, tree in (("a genuine BST", valid), ("6 under 15, under 10", sneaky)):
    print(f"{label:<26}{str(is_bst_wrong(tree)):>8}"
          f"{str(is_bst(tree)):>8}{str(is_bst_inorder(tree)):>10}")

print("\n🔴 The naive check declares the second tree valid. It is not:")
print("   in-order gives", inorder(sneaky), "- not sorted.")
print("\n   Every parent/child pair is individually fine. The rule is about")
print("   entire SUBTREES, not immediate children - which is exactly what")
print("   the range version enforces.")

## Three more standard questions

| Question | Key idea |
|---|---|
| **Diameter** — longest path between any two nodes | at each node it is `height(left) + height(right) + 2`; compute height and diameter in **one** pass |
| **Lowest common ancestor** — deepest node with both below it | in a **BST**, the first node whose value lies between them; in a general tree, recurse and see which side each is found on |
| **Balanced?** — is every subtree's height difference ≤ 1 | return height *and* balance together, so it is O(n) not O(n²) |

🔴 All three have a naive O(n²) version that recomputes height at every node, and an O(n) version that returns extra information up the recursion. **Returning a tuple instead of one value** is the technique — and interviewers are looking for it.

In [ ]:
def diameter(root):
    """Longest path between any two nodes, in edges. O(n): height and
    diameter are computed in the SAME pass.
    """
    best = 0

    def walk(node):
        nonlocal best
        if node is None:
            return -1                        # height of an empty tree
        left = walk(node.left)
        right = walk(node.right)
        best = max(best, left + right + 2)   # path THROUGH this node
        return 1 + max(left, right)          # height of this node

    walk(root)
    return best


def is_balanced(root):
    """O(n): returns (balanced, height) together rather than recomputing."""

    def walk(node):
        if node is None:
            return True, -1
        left_ok, left_h = walk(node.left)
        if not left_ok:
            return False, 0                  # short-circuit
        right_ok, right_h = walk(node.right)
        if not right_ok:
            return False, 0
        return abs(left_h - right_h) <= 1, 1 + max(left_h, right_h)

    return walk(root)[0]


def lca_bst(root, a, b):
    """In a BST: the first node whose value sits between the two. O(h)."""
    node = root
    low, high = min(a, b), max(a, b)
    while node is not None:
        if node.value > high:
            node = node.left
        elif node.value < low:
            node = node.right
        else:
            return node                      # the split point
    return None


tree = fresh_tree()
print("tree in-order:", inorder(tree))
print("height       :", height(tree))
print("diameter     :", diameter(tree), "edges")
print("balanced     :", is_balanced(tree))

print()
for a, b in ((20, 40), (20, 80), (60, 80), (30, 30)):
    ancestor = lca_bst(tree, a, b)
    print(f"  LCA({a:>2}, {b:>2}) = {ancestor.value}")

# a deliberately unbalanced tree
lopsided = TreeNode(1, TreeNode(2, TreeNode(3, TreeNode(4))))
print(f"\n  a left-leaning chain: balanced = {is_balanced(lopsided)}, "
      f"height = {height(lopsided)}")

## Interview questions

**1. Implement the four traversals.** *(above)*
> Know all four recursively, and in-order iteratively. Say that DFS uses a stack and BFS a queue.

**2. Validate a BST.** *(above)*
> 🔴 The range version, not the parent/child check. Volunteering the counter-example yourself is a strong signal.

**3. Maximum depth / height of a tree.**
> One line recursively. Clarify whether they count nodes or edges — off-by-one here is common and easily avoided by asking.

**4. Diameter of a binary tree.** *(above)*
> Compute height and diameter in one pass. The naive version is O(n²).

**5. Lowest common ancestor.** *(above)*
> In a BST, O(h) by comparing values. In a general binary tree, recurse: if both sides return non-None, this node is the LCA.

**6. Is the tree balanced?** *(above)*
> Return `(balanced, height)` together for O(n).

**7. Serialise and deserialise a binary tree.**
> Pre-order with explicit `None` markers. Level-order works too. In-order **alone cannot** reconstruct the tree — a good detail to raise.

**8. Convert a sorted array into a balanced BST.**
> Take the middle as the root, recurse on each half. O(n), and it is exactly the structure of binary search (**14.11**).

**9. Kth smallest element in a BST.**
> In-order traversal, stopping at k. O(h + k) with the iterative version, and no need to materialise the whole traversal.

**10. Why might a BST perform badly, and what would you do?**
> Sorted input degenerates it to a linked list, O(n) per operation. Use a self-balancing tree, shuffle the input, or use a hash map if you do not need ordering.

**11. Symmetric tree / mirror.**
> Recurse on `(left.left, right.right)` and `(left.right, right.left)` together.

**12. Path sum — does a root-to-leaf path total k?**
> DFS subtracting as you descend; check at the leaf. Watch the empty-tree case.

In [ ]:
# Questions 7, 8 and 9 - all commonly asked, all short.
def serialise(root):
    """Pre-order with None markers. Unambiguous, unlike in-order alone."""
    out = []

    def walk(node):
        if node is None:
            out.append("#")
            return
        out.append(str(node.value))
        walk(node.left)
        walk(node.right)

    walk(root)
    return ",".join(out)


def deserialise(text):
    tokens = iter(text.split(","))

    def build():
        token = next(tokens)
        if token == "#":
            return None
        node = TreeNode(int(token))
        node.left = build()
        node.right = build()
        return node

    return build()


def sorted_array_to_bst(values):
    """Middle becomes the root -> perfectly balanced. O(n)."""
    if not values:
        return None
    middle = len(values) // 2
    return TreeNode(values[middle],
                    sorted_array_to_bst(values[:middle]),
                    sorted_array_to_bst(values[middle + 1:]))


def kth_smallest(root, k):
    """Iterative in-order, stopping early. O(h + k), not O(n)."""
    stack, node, seen = [], root, 0
    while stack or node is not None:
        while node is not None:
            stack.append(node)
            node = node.left
        node = stack.pop()
        seen += 1
        if seen == k:
            return node.value
        node = node.right
    return None


tree = fresh_tree()
encoded = serialise(tree)
print("serialised   :", encoded)
restored = deserialise(encoded)
print("round-trips  :", inorder(restored) == inorder(tree))

values = list(range(1, 16))
built = sorted_array_to_bst(values)
print(f"\n{len(values)} sorted values -> BST of height {height(built)}")
print(f"  (inserting them in order would give height {len(values) - 1})")
print("  in-order matches the input:", inorder(built) == values)

print()
for k in (1, 4, 7):
    print(f"  {k}th smallest in {inorder(tree)} = {kth_smallest(tree, k)}")
print("  8th smallest (out of range):", kth_smallest(tree, 8))

---

## Common Mistakes & Pitfalls

1. 🔴 **Validating a BST by comparing only parent and child.** The rule constrains whole subtrees. Carry a range down, or check that in-order is strictly increasing.
2. 🔴 **Assuming a BST stays balanced.** Sorted input degenerates it into a linked list and every operation becomes O(n).
3. 🔴 **Recursing on a deep tree.** Height can be O(n); the recursion limit is ~1000. Use the iterative traversal.
4. **Confusing depth and height.** Depth counts down from the root, height counts up from the leaves.
5. **Recomputing height at every node** in diameter or balance checks - that is O(n²). Return a tuple and do it in one pass.
6. **Pushing the left child first** in an iterative pre-order. A stack reverses the order, so push right first.
7. **Forgetting the two-children case in delete**, or using the wrong replacement. It must be the in-order successor or predecessor.
8. **Using `list.pop(0)` as the BFS queue.** O(n) per operation - use `deque` (**14.5**).
9. **Believing in-order alone can rebuild a tree.** It cannot; you need pre-order or level-order with null markers.

## Best Practices

- Write tree code recursively first - a tree is defined recursively, so the code should be too.
- Ask whether height is counted in nodes or edges before you start.
- Use `deque` for BFS and a list for DFS.
- Snapshot `len(queue)` before draining, when you need per-level results.
- Return tuples from recursive helpers to avoid recomputing - the single most common O(n²)→O(n) fix in tree problems.
- Test: empty tree, a single node, a left-leaning chain, and a right-leaning chain.
- In production Python, reach for `dict` (**14.6**) unless you specifically need sorted order.
- Know what a rotation does and why balanced trees exist; do not memorise a red-black insert.

## Practice Exercises

Try these before moving on.

1. Implement `lca` for a **general** binary tree (no BST property). Why is it O(n) rather than O(h)?
2. 🔴 Write `is_bst_wrong` from memory, then construct your own counter-example that defeats it. Do this before reading the answer again.
3. Implement level-order traversal *without* a queue, using recursion and a depth parameter. Which version would you rather maintain?
4. Implement `kth_largest` by running the in-order traversal in reverse, and verify it against `sorted(inorder(tree))[-k]`.
5. Write `insert` iteratively rather than recursively, so it works on a degenerate tree of 100,000 nodes.
6. Implement AVL rotation logic: detect when a node's subtrees differ in height by more than 1, and apply the correct single or double rotation.
7. 🔴 Build a BST from 10,000 sorted values and one from 10,000 shuffled values. Time 10,000 searches in each and explain the ratio using **14.1**.
8. Serialise a tree with level-order plus null markers instead of pre-order, and write the matching deserialiser.